### Explicación del Código

El código de esta celda configura un motor de análisis y anonimización de texto utilizando Presidio y un modelo NER personalizado. Además, incluye funciones para generar reportes en formato TXT y HTML. Realiza las siguientes tareas:

1. **Configuración del modelo NER**:
   - Carga un modelo NER personalizado desde la ruta `/home/sotavento/Documents/tejer_red/NER/3_prueba_modelo/model-best`.
   - Define etiquetas NER confirmadas como `NOMBRE`, `DOMICILIO`, `FECHA`, `HORA`.

2. **Definición de patrones Regex**:
   - Añade reconocedores para entidades como RFC, CURP, teléfonos, placas, fechas, horas, domicilios y colonias.

3. **Inicialización de Presidio**:
   - Configura el motor `AnalyzerEngine` con el modelo NER y los patrones Regex.
   - Define operadores de anonimización para reemplazar entidades detectadas con valores protegidos (e.g., `[NOMBRE PROTEGIDO]`).

4. **Funciones de reporte**:
   - **TXT**: Genera un archivo de texto con los resultados del análisis, incluyendo las entidades detectadas y los textos anonimizados.
   - **HTML**: Genera un reporte visual con Bootstrap, resaltando las entidades detectadas y mostrando los textos anonimizados.

#### Archivos y Carpetas Relevantes:
- **Modelo NER**: `/home/sotavento/Documents/tejer_red/NER/3_prueba_modelo/model-best`.
- **Salida**:
  - Reporte TXT: `reporte_anonimizacion.txt`.
  - Reporte HTML: `reporte_anonimizacion_bootstrap.html`.

In [ ]:
# Instalar dependencias necesarias
!pip install pandas spacy presidio-analyzer presidio-anonymizer tqdm

In [ ]:
# --- Celda 1: CONFIGURACIÓN FINAL PRESIDIO (V12) + FUNCIONES DE REPORTE ---
# Fecha/Hora: Jueves, 10 de abril de 2025, 5:20 PM CST
# Ubicación: Guadalajara, Jalisco, México
# Descripción: Configura Analyzer V12 (custom NER + Regex con compromisos)
#              Define operadores de anonimización.
#              Define funciones para generar reportes TXT y HTML (con Bootstrap).

import pandas as pd
import spacy
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_analyzer.predefined_recognizers import SpacyRecognizer
# Importar otros defaults que SÍ funcionan si quieres analizarlos también
from presidio_analyzer.predefined_recognizers import PhoneRecognizer, EmailRecognizer, DateRecognizer, UrlRecognizer
# (Añadir más si los necesitas: IpRecognizer, EsNifRecognizer, etc.)
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import os
from IPython.display import display, Markdown, HTML # Importar HTML para visualización opcional en notebook
import re
import traceback
import random # Para el muestreo

print("--- Iniciando Celda de Configuración V12 + Reportes ---")

# --- PASO 1: Ruta al modelo NER custom ---
ruta_modelo_custom = "/home/sotavento/Documents/tejer_red/NER/3_prueba_modelo/model-best"

# --- PASO 2: Etiquetas NER confirmadas ---
lista_etiquetas_NER_confirmada = ["NOMBRE", "DOMICILIO", "FECHA", "HORA"]
print(f"Etiquetas NER del modelo: {lista_etiquetas_NER_confirmada}")

# --- PASO 3: Carga engine ---
nlp_engine_es_only = None
analyzer_v12 = None

if not os.path.exists(ruta_modelo_custom):
    print(f"--- ADVERTENCIA: Ruta no encontrada: {ruta_modelo_custom} ---")
else:
    print(f"Ruta del modelo encontrada: {ruta_modelo_custom}")
    nlp_config_es_only = {"nlp_engine_name": "spacy", "models": [{"lang_code": "es", "model_name": ruta_modelo_custom}]}
    try:
        print("\nCargando motor NLP...")
        provider_es_only = NlpEngineProvider(nlp_configuration=nlp_config_es_only)
        nlp_engine_es_only = provider_es_only.create_engine()
        print(f"Motor NLP cargado.")

        # --- PASO 4: Inicialización Analyzer V12 ---
        print("\nInicializando AnalyzerEngine (V12)...")
        analyzer_v12 = AnalyzerEngine(nlp_engine=nlp_engine_es_only, supported_languages=["es"])
        print("AnalyzerEngine V12 creado.")

        # 4.b SpacyRecognizer custom
        custom_spacy_recognizer = SpacyRecognizer(supported_language="es", supported_entities=lista_etiquetas_NER_confirmada)
        analyzer_v12.registry.add_recognizer(custom_spacy_recognizer)
        print(f"- SpacyRecognizer personalizado añadido (para {lista_etiquetas_NER_confirmada}).")

        # --- Crear y Añadir Recognizers Regex ---
        # RFC (Simple)
        rfc_patterns_simple = [
            Pattern(name="RFC Simple Pattern", regex=r"\b[A-Z&Ñ]{3,4}\d{6}[A-Z\d]{3}\b", score=0.6)
        ]
        rfc_recognizer_simple = PatternRecognizer(
            supported_entity="MX_RFC",
            patterns=rfc_patterns_simple,
            name="Mexican RFC Recognizer Simple",
            supported_language="es"
        )
        analyzer_v12.registry.add_recognizer(rfc_recognizer_simple)

        # CURP (Simple)
        curp_patterns_simple = [
            Pattern(name="CURP Simple Pattern", regex=r"\b[A-Z]{4}\d{6}[HM][A-Z\d]{7}\b", score=0.7)
        ]
        curp_recognizer_simple = PatternRecognizer(
            supported_entity="MX_CURP",
            patterns=curp_patterns_simple,
            name="Mexican CURP Recognizer Simple",
            supported_language="es"
        )
        analyzer_v12.registry.add_recognizer(curp_recognizer_simple)

        # EXP (Expediente)
        expediente_patterns = [
            #Pattern(name="Expediente Pattern", regex=r"\b(?:EXP\.|EXPEDIENTE)\s+\d{1,5}/\d{4}-\d{2}\b", score=0.7),
            Pattern(name="Case Number NNNN/NNNN-NN", regex=r"\b\d{4}/\d{4}-\d{2}\b", score=0.75)
        ]
        expediente_recognizer = PatternRecognizer(
            supported_entity="EXP",
            patterns=expediente_patterns,
            name="Expediente Recognizer",
            supported_language="es",
            context=["exp", "cobupej", "expediente"]

            
        )
        analyzer_v12.registry.add_recognizer(expediente_recognizer)

        # TELEFONO
        telefono_patterns = [
            Pattern(name="Telefono 10 digitos", regex=r"\b\d{10}\b", score=0.6),
            Pattern(name="Telefono 10 digitos con guiones", regex=r"\b\d{2}(?:[.\s-]\d{2}){4}\b", score=0.6),
            Pattern(name="Number 9 Digit", regex=r"\b\d{9}\b", score=0.6)
        ]
        telefono_recognizer_regex = PatternRecognizer(
            supported_entity="TELEFONO",
            patterns=telefono_patterns,
            name="Telefono Regex Recognizer",
            supported_language="es",
            context=["tel", "telefono", "celular", "movil", "llamar", "whatsapp", "contacto", "numero"]
        )
        analyzer_v12.registry.add_recognizer(telefono_recognizer_regex)

        # PLACA (License Plate)
        license_plate_patterns_final = [
            #Pattern(name="Placa AAA NNNA context", regex=r"\b[A-Z]{3}(?:-|\s)?\d{3}[A-Z]\b", score=0.7),
            Pattern(name="Placa AAA-NN-NN", regex=r"\b[A-Z]{3}-\d{2}-\d{2}\b", score=0.8),
            Pattern(name="Placa ANN-AAA", regex=r"\b[A-Z]\d{2}-[A-Z]{3}\b", score=0.8),
            Pattern(name="Placa NANAA", regex=r"\b\d[A-Z]\d[A-Z]{2}\b", score=0.7),
            Pattern(name="Placa AAA-NNN-A", regex=r"\b[A-Z]{3}-\d{3}-[A-Z]\b", score=0.8),
            Pattern(name="Placa ANAAA", regex=r"\b[A-Z]\d[A-Z]{3}\b", score=0.7),
            Pattern(name="Placa ANANNNNA", regex=r"\b[A-Z]\d[A-Z]\d{3}[A-Z]\b", score=0.8),
            Pattern(name="Placa NNAAAN", regex=r"\b\d{2}[A-Z]{3}\d\b", score=0.75), # Para 88NVN4
            Pattern(name="Placa AA-NN-NNN", regex=r"\b[A-Z]{2}-\d{2}-\d{3}\b", score=0.75), # Para JP-34-528
            Pattern(name="Placa LLLNLN context", regex=r"\b[A-Z]{3}\d[A-Z]\b", score=0.75), # Matches PEV8R, SXC2G
            Pattern(name="Placa LLNNNNN context", regex=r"\b[A-Z]{2}\d{5}\b", score=0.75), # Matches RY98392, JT98564

            # --- Added patterns for recently missed plates ---
            #Pattern(name="Placa LLL NNN context", regex=r"\b[A-Z]{3}\s\d{3}\b", score=0.75), # Matches LKW 026
            Pattern(name="Placa LNLNNNN context", regex=r"\b[A-Z]\d[A-Z]\d{4}\b", score=0.75), # Matches J5W7615
            Pattern(name="Placa LLL-NNNN context", regex=r"\b[A-Z]{3}-\d{4}\b", score=0.75)  # Matches JFS-6803, JFJ-6903
        ]
        plate_recognizer_final = PatternRecognizer(
            supported_entity="PLACA",
            patterns=license_plate_patterns_final,
            name="PlateRecognizer_Final_Compromise",
            supported_language="es",
            context=["placa", "placas", "matricula", "matrícula", "vehiculo", "vehículo", "auto", "automovil", "camioneta", "moto", "motocicleta", "coche"]
        )
        analyzer_v12.registry.add_recognizer(plate_recognizer_final)

        # --- Crear y Añadir Recognizers Regex basados en Reglas EntityRuler ---
        print("\nAñadiendo recognizers Regex adicionales (Fecha, Hora, Domicilio, Colonia)...")

        # 7. Recognizer para FECHA (Regla Específica)
        fecha_rule_patterns = [
            Pattern(name="Fecha Día Mes Año", regex=r"\b[Dd][íÍ][aA]\s+\d{1,2}\s+de\s+(?:enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre)\s+del\s+\d{4}\b", score=0.9)
        ]
        fecha_rule_recognizer = PatternRecognizer(supported_entity="FECHA", patterns=fecha_rule_patterns, name="Fecha Rule Recognizer", supported_language="es")
        analyzer_v12.registry.add_recognizer(fecha_rule_recognizer)
        print(f"- '{fecha_rule_recognizer.name}' añadido.")

        # 8. Recognizer para HORA (Regla Específica)
        hora_rule_patterns = [
            Pattern(name="Hora HH:MM (AM/PM)?", regex=r"\b\d{1,2}:\d{2}\s*(?:[AaPp]\.?[Mm]\.?)?\b", score=0.85)
        ]
        hora_rule_recognizer = PatternRecognizer(supported_entity="HORA", patterns=hora_rule_patterns, name="Hora Rule Recognizer", supported_language="es")
        analyzer_v12.registry.add_recognizer(hora_rule_recognizer)
        print(f"- '{hora_rule_recognizer.name}' añadido.")

        # 9. Recognizer para DOMICILIO (Regla Específica - PRECAUCIÓN)
        #    Este Regex es limitado y puede fallar o dar FPs. NER es preferible.
        domicilio_rule_patterns = [
            Pattern(name="Domicilio Calle Nombre #Num", regex=r"\b(?:[Cc][aA][lL]{2}[eE]|[Aa][vV][eE][nN][iI][dD][aA]|Av)\.?\s+[A-ZÁÉÍÓÚÑa-záéíóúñ]+\s+(?:(?:#|N[uU][mM][eE]?[rR]?[oO]?\.?)\s?\d+)\b", score=0.75)
        ]
        domicilio_rule_recognizer = PatternRecognizer(supported_entity="DOMICILIO", patterns=domicilio_rule_patterns, name="Domicilio Rule Recognizer", supported_language="es")
        analyzer_v12.registry.add_recognizer(domicilio_rule_recognizer)
        print(f"- '{domicilio_rule_recognizer.name}' añadido (Precaución: Regex limitado).")

        # 10. Recognizer para COLONIA (Regla Específica - PRECAUCIÓN)
        #     Usaremos la etiqueta DOMICILIO también o una nueva COLONIA? Usemos DOMICILIO por ahora.
        #     Este Regex es limitado. NER es preferible.
        colonia_rule_patterns = [
            Pattern(name="Colonia Nombre Palabra+", regex=r"\b(?:[Cc][oO][lL]\.|[Cc][oO][lL][oO][nN][iI][aA])\s+[A-ZÁÉÍÓÚÑ][A-ZÁÉÍÓÚÑa-záéíóúñ]+(?:\s+[A-ZÁÉÍÓÚÑa-záéíóúñ]+)*\b", score=0.6)
        ]
        colonia_rule_recognizer = PatternRecognizer(supported_entity="COLONIA", patterns=colonia_rule_patterns, name="Colonia Rule Recognizer", supported_language="es")
        analyzer_v12.registry.add_recognizer(colonia_rule_recognizer)
        print(f"- '{colonia_rule_recognizer.name}' añadido (Precaución: Regex limitado).")


        print("- Recognizers Regex (RFC, CURP, EXP, TELEFONO, PLACA) añadidos.")

        print("\n--- CONFIGURACIÓN DEL ANALYZER V12 (FINAL) COMPLETA ---")

    except Exception as e:
        print(f"\n--- ERROR DURANTE LA CARGA O INICIALIZACIÓN V12 ---")
        print(f"Error: {e}")
        traceback.print_exc()
        analyzer_v12 = None

# --- PASO 5: Definir Configuración de Anonimización ---
operators_config_final = { "DEFAULT": OperatorConfig("replace", {"new_value": "[?]"}), "NOMBRE": OperatorConfig("replace", {"new_value": "[NOMBRE PROTEGIDO]"}), "DOMICILIO": OperatorConfig("replace", {"new_value": "[DOMICILIO PROTEGIDO]"}), "COLONIA": OperatorConfig("replace", {"new_value": "[COLONIA PROTEGIDA]"}), "FECHA": OperatorConfig("replace", {"new_value": "[FECHA PROTEGIDA]"}), "HORA": OperatorConfig("replace", {"new_value": "[HORA PROTEGIDA]"}), "EXP": OperatorConfig("replace", {"new_value": "[EXP PROTEGIDO]"}), "TELEFONO": OperatorConfig("replace", {"new_value": "[TELEFONO PROTEGIDO]"}), "PLACA": OperatorConfig("replace", {"new_value": "[PLACA PROTEGIDA]"}), "MX_RFC": OperatorConfig("replace", {"new_value": "[RFC PROTEGIDO]"}), "MX_CURP": OperatorConfig("replace", {"new_value": "[CURP PROTEGIDO]"}), }
anonymizer = AnonymizerEngine()
print("\nAnonymizerEngine y Operadores definidos.")

# --- PASO 6: Definir Variables para Análisis ---
# Lista COMPLETA de entidades a buscar (NER Confirmadas + Regex)
#lista_etiquetas_REGEX = ["TELEFONO", "EXP", "PLACA", "MX_RFC", "MX_CURP", "COLONIA"]
lista_etiquetas_REGEX = ["TELEFONO", "EXP", "PLACA", "MX_RFC", "MX_CURP"]
entidades_a_buscar_final = list(set(lista_etiquetas_NER_confirmada + lista_etiquetas_REGEX))
umbral_confianza = 0.1 # Umbral de análisis
print(f"\nConfiguración de análisis: Umbral={umbral_confianza}, Entidades={sorted(entidades_a_buscar_final)}")

# --- PASO 7: Funciones de Reporte (Adaptadas del TXT) ---

# 7.a Colores para resaltar (puedes ajustar)
entity_colors = { "NOMBRE": "#CCE5FF", "DOMICILIO": "#FFCCCC","COLONIA": "#FFC5CC", "FECHA": "#FFFFCC", "HORA": "#D9CCFF", "EXP": "#FFCC99", "TELEFONO": "#CCFFCC", "PLACA": "#FF99CC", "MX_RFC": "#99CCFF", "MX_CURP": "#FFCCFF",}

# 7.b Función para resaltar (igual que en el TXT) [cite: 2]
def highlight_entities(text, analysis_results):
    highlighted_text = text
    offset = 0
    # Ordenar por inicio para aplicar cambios secuencialmente
    for result in sorted(analysis_results, key=lambda x: x.start):
        start = result.start + offset
        end = result.end + offset
        entity_type = result.entity_type
        color = entity_colors.get(entity_type, "#FFFFFF") # Blanco por defecto [cite: 2]

        # Crear el span HTML [cite: 3, 4]
        span = f'<span style="background-color: {color}; padding: 2px 4px; border-radius: 3px; border: 1px solid #ccc;">{highlighted_text[start:end]}<sup style="font-size: 0.7em; color: #555; margin-left: 2px;">{entity_type}</sup></span>'
        highlighted_text = highlighted_text[:start] + span + highlighted_text[end:]
        offset += len(span) - (end - start)
    return highlighted_text

# 7.c Función para generar reporte HTML con BOOTSTRAP
def generate_html_report_bootstrap(texts, analysis_results_list, anonymized_texts, output_file="reporte_anonimizacion_bootstrap.html"):
    print(f"\nGenerando reporte HTML con Bootstrap: {output_file}")
    html_content = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Reporte de Anonimización Presidio</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet" integrity="sha384-QWTKZyjpPEjISv5WaRU9OFeRpok6YctnYmDr5pNlyT2bRjXh0JMhjY6hW+ALEwIH" crossorigin="anonymous">
    <style>
        body { padding-top: 20px; }
        .card { margin-bottom: 1.5rem; }
        .card-body p { margin-bottom: 0.5rem; }
        .original-text, .highlighted-text, .anonymized-text {
             white-space: pre-wrap; /* Conservar saltos de línea */
             word-wrap: break-word; /* Evitar desbordamiento */
             font-family: monospace; /* Mejor para ver texto original */
             font-size: 0.9rem;
             padding: 10px;
             border: 1px solid #eee;
             background-color: #f9f9f9;
             border-radius: 4px;
        }
        .anonymized-text { font-weight: bold; background-color: #e9e9e9; }
    </style>
</head>
<body>
    <div class="container">
        <h1 class="mb-4">Reporte de Anonimización Presidio</h1>
"""

    if not texts:
        html_content += '<p class="text-muted">No hay textos para reportar.</p>'

    for i, (text, analysis_results, anonymized_text) in enumerate(zip(texts, analysis_results_list, anonymized_texts)):
        try:
            # Generar texto resaltado SOLO si hay resultados
            highlighted_version = highlight_entities(text, analysis_results) if analysis_results else text
        except Exception as high_err:
            print(f"  - Advertencia: Error al resaltar texto #{i+1}: {high_err}")
            highlighted_version = f"{text} (Error al resaltar)"

        html_content += f"""
        <div class="card shadow-sm">
            <div class="card-header">
                Texto Ejemplo #{i + 1}
            </div>
            <div class="card-body">
                <h6 class="card-subtitle mb-2 text-muted">Original:</h6>
                <p class="original-text">{text}</p>
                <hr>
                <h6 class="card-subtitle mb-2 text-muted">Resaltado (Entidades Detectadas):</h6>
                <p class="highlighted-text">{highlighted_version}</p>
                <hr>
                <h6 class="card-subtitle mb-2 text-muted">Anonimizado:</h6>
                <p class="anonymized-text">{anonymized_text}</p>
            </div>
        </div>
        """

    html_content += """
    </div>
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/js/bootstrap.bundle.min.js" integrity="sha384-YvpcrYf0tY3lHB60NNkmXc5s9fDVZLESaAA55NDzOxhy9GkcIdslK1eN7N6jIeHz" crossorigin="anonymous"></script>
</body>
</html>
"""
    try:
        with open(output_file, "w", encoding="utf-8") as file:
            file.write(html_content) # [cite: 12]
        print(f"Reporte HTML generado exitosamente: {output_file}")
    except Exception as write_err:
        print(f"Error al escribir el archivo HTML '{output_file}': {write_err}")

# 7.d Función para generar reporte TXT
def generate_txt_report(texts, analysis_results_list, anonymized_texts, output_file="reporte_anonimizacion.txt"):
    print(f"\nGenerando reporte TXT: {output_file}")
    try:
        with open(output_file, "w", encoding="utf-8") as file:
            if not texts:
                file.write("No hay textos para reportar.\n")
                return

            for i, (text, analysis_results, anonymized_text) in enumerate(zip(texts, analysis_results_list, anonymized_texts)):
                file.write(f"--- Texto Ejemplo #{i+1} ---\n")
                file.write("Original:\n")
                file.write(text + "\n\n")
                file.write("Entidades Detectadas:\n")
                if analysis_results:
                     for res in analysis_results:
                         # Cuidado aquí si analysis_explanation es None
                         score = f"{res.score:.2f}"
                         file.write(f" - {res.entity_type}: '{text[res.start:res.end]}' (Score: {score}, Pos: {res.start}-{res.end})\n")
                else:
                     file.write("  (Ninguna)\n")
                file.write("\nAnonimizado:\n")
                file.write(anonymized_text + "\n")
                file.write("-" * 40 + "\n\n")
        print(f"Reporte TXT generado exitosamente: {output_file}")
    except Exception as write_err:
        print(f"Error al escribir el archivo TXT '{output_file}': {write_err}")


print("\n--- Funciones de Reporte Definidas ---")

### Explicación del Código

El código en esta celda realiza el análisis y anonimización de textos extraídos de un archivo CSV utilizando Presidio. Además, genera reportes en formato TXT y HTML. Las tareas principales son:

1. **Configuración de Entrada/Salida**:
   - **Archivo CSV**: `repd_vp_cedulas_principal.csv`.
   - **Columna a procesar**: `descripcion_desaparicion`.
   - **Número de muestras**: 3 (puede ajustarse).
   - **Archivos de salida**:
     - Reporte TXT: `reporte_anonimizacion.txt`.
     - Reporte HTML: `reporte_anonimizacion_visual.html`.

2. **Verificación de Configuración**:
   - Comprueba que el motor de análisis (`analyzer_v12`), el anonimizador (`anonymizer`), y las configuraciones necesarias están definidas.

3. **Lectura y Muestreo del CSV**:
   - Lee el archivo CSV y limpia valores nulos en la columna especificada.
   - Toma una muestra aleatoria de textos (ajustando el número de muestras si hay menos datos disponibles).

4. **Procesamiento de Textos**:
   - **Análisis**: Detecta entidades sensibles (e.g., nombres, domicilios, fechas) utilizando el motor de análisis.
   - **Anonimización**: Reemplaza las entidades detectadas con valores protegidos (e.g., `[NOMBRE PROTEGIDO]`).

5. **Generación de Reportes**:
   - **TXT**: Incluye los textos originales, las entidades detectadas y los textos anonimizados.
   - **HTML**: Reporte visual con Bootstrap, resaltando las entidades detectadas y mostrando los textos anonimizados.

#### Archivos y Carpetas Relevantes:
- **Entrada**: `repd_vp_cedulas_principal.csv`.
- **Salida**:
  - `reporte_anonimizacion.txt`.
  - `reporte_anonimizacion_visual.html`.


In [ ]:
# --- Celda 2: Carga CSV, Muestreo, Proceso y Reportes ---
# Fecha/Hora: Jueves, 10 de abril de 2025, 5:20 PM CST
# Ubicación: Guadalajara, Jalisco, México

import pandas as pd
import random
import os  # Ya importado, pero por claridad
from tqdm import tqdm  # Para la barra de progreso

# --- PASO 1: Configuración de Entrada/Salida ---
# ¡¡MODIFICA ESTOS VALORES!!
csv_file_path = "repd_vp_cedulas_principal.csv"  # <-- Poner la ruta correcta a tu CSV
column_name = "descripcion_desaparicion"  # <-- Nombre de la columna con los textos
num_samples = 3  # <-- Cuántos casos procesar (ej. 5, 10, 20...)
output_txt_file = "reporte_anonimizacion.txt"
output_html_file = "reporte_anonimizacion_visual.html"
random_seed = 594  # Opcional: para que el muestreo aleatorio sea repetible

print(f"Archivo CSV a procesar: {csv_file_path}")
print(f"Columna a procesar: {column_name}")
print(f"Número de muestras aleatorias: {num_samples}")

# --- PASO 2: Verificar que el Analyzer V12 está listo ---
if 'analyzer_v12' not in locals() or not analyzer_v12:
    print("\nERROR: El objeto 'analyzer_v12' no está definido. Ejecuta la Celda 1 primero.")
    raise NameError("Analyzer V12 no está listo.")
if 'anonymizer' not in locals():
    print("\nERROR: El objeto 'anonymizer' no está definido. Ejecuta la Celda 1 primero.")
    raise NameError("Anonymizer no está listo.")
if 'operators_config_final' not in locals():
    print("\nERROR: 'operators_config_final' no definido. Ejecuta la Celda 1 primero.")
    raise NameError("Operators Config no está listo.")
if 'entidades_a_buscar_final' not in locals():
    print("\nERROR: 'entidades_a_buscar_final' no definido. Ejecuta la Celda 1 primero.")
    raise NameError("Lista de Entidades no está lista.")
if 'umbral_confianza' not in locals():
    umbral_confianza = 0.1  # Definir si falta
    print(f"ADVERTENCIA: umbral_confianza no definido, usando {umbral_confianza}")

# --- PASO 3: Leer CSV y Tomar Muestra Aleatoria ---
texts_to_process = []
sampled_df = None

try:
    print(f"\nLeyendo archivo CSV: {csv_file_path}")
    df = pd.read_csv(csv_file_path)
    print(f"CSV leído. Total de filas: {len(df)}")

    if column_name not in df.columns:
        print(f"ERROR: La columna '{column_name}' no se encontró en el CSV.")
        raise ValueError(f"Columna no encontrada: {column_name}")

    # Limpiar textos (quitar NaN y convertir a string)
    df_cleaned = df.dropna(subset=[column_name])
    df_cleaned[column_name] = df_cleaned[column_name].astype(str)
    print(f"Filas después de limpiar NAs en '{column_name}': {len(df_cleaned)}")

    if len(df_cleaned) == 0:
        print("No hay datos válidos en la columna especificada.")
    else:
        # Ajustar num_samples si es mayor que las filas disponibles
        actual_samples = min(num_samples, len(df_cleaned))
        if actual_samples < num_samples:
            print(f"Advertencia: Se solicitaron {num_samples} muestras, pero solo hay {actual_samples} filas válidas.")

        print(f"Tomando {actual_samples} muestras aleatorias...")
        sampled_df = df_cleaned.sample(n=actual_samples, random_state=random_seed)
        texts_to_process = sampled_df[column_name].tolist()
        print("Muestras obtenidas.")

except FileNotFoundError:
    print(f"ERROR: Archivo CSV no encontrado en la ruta: {csv_file_path}")
    sampled_df = pd.DataFrame()  # DataFrame vacío para evitar errores después
    texts_to_process = []
except Exception as e:
    print(f"ERROR al leer o muestrear el CSV: {e}")
    sampled_df = pd.DataFrame()
    texts_to_process = []

# --- PASO 4: Procesar los Textos Muestreados (Análisis y Anonimización) ---
anonymized_texts_sampled = []
analysis_results_sampled = []

if not texts_to_process:
    print("\nNo hay textos para procesar debido a errores previos o datos insuficientes.")
else:
    print("\n--- Iniciando Procesamiento de Muestras ---")
    for i, texto in enumerate(tqdm(texts_to_process, desc="Procesando muestras")):
        try:
            # --- ANÁLISIS ---
            resultados_analisis = analyzer_v12.analyze(
                text=texto,
                language="es",
                score_threshold=umbral_confianza,
                entities=entidades_a_buscar_final,
                return_decision_process=True
            )
            analysis_results_sampled.append(resultados_analisis)

            # --- ANONIMIZACIÓN ---
            resultado_anonimizado = anonymizer.anonymize(
                text=texto,
                analyzer_results=resultados_analisis,
                operators=operators_config_final
            )
            anonymized_texts_sampled.append(resultado_anonimizado.text)

        except Exception as e:
            anonymized_texts_sampled.append(f"ERROR AL PROCESAR: {e}")
            analysis_results_sampled.append([])

    print("--- Fin del Procesamiento de Muestras ---")

    # --- PASO 5: Generar Reportes ---
    if len(texts_to_process) == len(analysis_results_sampled) == len(anonymized_texts_sampled):
        # Generar Reporte TXT
        generate_txt_report(texts_to_process, analysis_results_sampled, anonymized_texts_sampled, output_txt_file)

        # Generar Reporte HTML con Bootstrap
        generate_html_report_bootstrap(texts_to_process, analysis_results_sampled, anonymized_texts_sampled, output_html_file)
    else:
        print("ERROR: Discrepancia en la longitud de las listas de resultados, no se pueden generar reportes.")

### Explicación del Código

El código genera un archivo JSON con anotaciones de entidades detectadas en textos procesados desde un archivo CSV utilizando Presidio. Realiza las siguientes tareas:

1. **Verificación de Configuración**:
   - Comprueba que el motor de análisis (`analyzer_v12`) y las configuraciones necesarias están definidos.

2. **Configuración de Entrada y Parámetros**:
   - **Archivo CSV**: `repd_vp_cedulas_principal.csv`.
   - **Columna a procesar**: `descripcion_desaparicion`.
   - **Número de muestras**: 20 (puede ajustarse).
   - **Archivo de salida**: `presidio_annotations_output.json`.
   - **Entidades a buscar**: Combina etiquetas NER confirmadas (e.g., `NOMBRE`, `DOMICILIO`) y patrones Regex (e.g., `TELEFONO`, `MX_RFC`).

3. **Lectura y Muestreo del CSV**:
   - Lee el archivo CSV, limpia valores nulos y toma una muestra aleatoria de textos para procesar.

4. **Procesamiento de Textos**:
   - Analiza los textos con Presidio para detectar entidades sensibles.
   - Formatea las entidades detectadas en el formato JSON: `[[start, end, label], ...]`.

5. **Generación del JSON**:
   - Crea un archivo JSON con las clases (etiquetas únicas) y las anotaciones de las entidades detectadas.
   - Guarda el archivo en `presidio_annotations_output.json`.

#### Archivos y Carpetas Relevantes:
- **Entrada**: `repd_vp_cedulas_principal.csv`.
- **Salida**: `presidio_annotations_output.json`.

#### Dependencias:
- Librerías: `pandas`, `random`, `json`, `presidio-analyzer`, `os`, `traceback`.

In [ ]:
# --- Celda 3: Generar JSON de Anotaciones desde Presidio ---
# Fecha/Hora: Jueves, 10 de abril de 2025, 5:35 PM CST
# Ubicación: Guadalajara, Jalisco, México
# Descripción: Lee textos de un CSV, toma una muestra, los analiza con
#              Presidio (Analyzer V12) y guarda las entidades detectadas
#              en un archivo JSON con formato específico.

import json
import pandas as pd
import random
import os
import traceback
from IPython.display import display, Markdown # Para mensajes en notebook

print("--- Iniciando Celda: Generación de JSON de Anotaciones ---")

# --- PASO 1: Verificar que Analyzer V12 está listo ---
if 'analyzer_v12' not in locals() or not analyzer_v12:
    print("ERROR FATAL: 'analyzer_v12' no está definido. Ejecuta la Celda 1 (Config V12) primero.")
    # Detener si no está listo
    raise NameError("Analyzer V12 (definido en Celda 1) no está listo.")
else:
    print("Analyzer 'analyzer_v12' encontrado y listo.")

# --- PASO 2: Configuración de Entrada y Parámetros ---
# ¡¡MODIFICA ESTOS VALORES SEGÚN NECESITES!!
csv_file_path = "repd_vp_cedulas_principal.csv"  # <-- Poner la ruta correcta a tu CSV
column_name = "descripcion_desaparicion" # <-- Nombre de la columna con los textos
num_samples = 20  # <-- Cuántas muestras aleatorias procesar para el JSON
random_seed = 123 # Opcional: para repetibilidad del muestreo
output_json_file = "presidio_annotations_output.json" # Nombre del archivo de salida

# Parámetros de análisis (deben coincidir con la Celda 1 de V12)
umbral_confianza = 0.1
if 'lista_etiquetas_NER_confirmada' not in locals(): lista_etiquetas_NER_confirmada = ["NOMBRE", "DOMICILIO", "FECHA", "HORA"]
if 'lista_etiquetas_REGEX' not in locals(): lista_etiquetas_REGEX = ["TELEFONO", "EXP", "PLACA", "MX_RFC", "MX_CURP"]
entidades_a_buscar_final = list(set(lista_etiquetas_NER_confirmada + lista_etiquetas_REGEX))

print(f"\nConfiguración:")
print(f"  - Archivo CSV: {csv_file_path}")
print(f"  - Columna: {column_name}")
print(f"  - Muestras: {num_samples}")
print(f"  - Archivo JSON Salida: {output_json_file}")
print(f"  - Entidades a buscar: {sorted(entidades_a_buscar_final)}")
print(f"  - Umbral de confianza: {umbral_confianza}")

# --- PASO 3: Leer CSV y Tomar Muestra Aleatoria ---
texts_to_process = []
try:
    if not os.path.exists(csv_file_path): raise FileNotFoundError(f"Archivo no encontrado en: {csv_file_path}")

    print(f"\nLeyendo archivo CSV: {csv_file_path}...")
    try: df = pd.read_csv(csv_file_path)
    except UnicodeDecodeError: df = pd.read_csv(csv_file_path, encoding='latin1') # Fallback encoding

    print(f"CSV leído. Total de filas: {len(df)}")
    if column_name not in df.columns: raise ValueError(f"Columna '{column_name}' no encontrada.")

    df_cleaned = df.dropna(subset=[column_name])
    df_cleaned[column_name] = df_cleaned[column_name].astype(str)
    print(f"Filas válidas en '{column_name}': {len(df_cleaned)}")

    if len(df_cleaned) == 0: print("Advertencia: No hay datos válidos para procesar.")
    else:
        actual_samples = min(num_samples, len(df_cleaned))
        if actual_samples < num_samples: print(f"Advertencia: Se tomarán {actual_samples} muestras (se pidieron {num_samples}).")
        else: print(f"Tomando {actual_samples} muestras aleatorias...")

        sampled_df = df_cleaned.sample(n=actual_samples, random_state=random_seed)
        texts_to_process = sampled_df[column_name].tolist()
        print(f"Se procesarán {len(texts_to_process)} textos.")

except FileNotFoundError as fnf_err: print(f"ERROR FATAL: {fnf_err}")
except ValueError as val_err: print(f"ERROR FATAL: {val_err}")
except Exception as e: print(f"ERROR FATAL al leer/muestrear CSV: {e}"); traceback.print_exc()

# --- PASO 4: Procesar Textos y Formatear Anotaciones ---
all_annotations_list = []
all_labels_set = set()

if texts_to_process:
    print("\nProcesando textos con Presidio Analyzer...")
    for i, text in enumerate(texts_to_process):
        # Imprimir progreso cada 10 textos si son muchos
        if (i + 1) % 10 == 0 or i == 0:
             print(f"  Procesando texto #{i+1} de {len(texts_to_process)}...")
        try:
            # Solo necesitamos el resultado del análisis
            results = analyzer_v12.analyze(
                text=text,
                language="es",
                score_threshold=umbral_confianza,
                entities=entidades_a_buscar_final
                # No necesitamos return_decision_process si solo queremos start/end/label
            )

            # Formatear entidades para este texto: [[start, end, label], ...]
            current_entities_json_format = []
            if results:
                for res in results:
                    # Añadir la entidad al formato [start, end, label]
                    current_entities_json_format.append([res.start, res.end, res.entity_type])
                    # Recolectar la etiqueta única
                    all_labels_set.add(res.entity_type)

            # Añadir al formato JSON principal: [texto, {"entities": [...]}]
            all_annotations_list.append([text, {"entities": current_entities_json_format}])

        except Exception as analyze_err:
            print(f"  ERROR al analizar texto #{i+1}: {analyze_err}")
            # Añadir texto sin entidades en caso de error para mantener la estructura
            all_annotations_list.append([text, {"entities": []}])

    print("Procesamiento completado.")
else:
    print("No hubo textos para procesar.")

# --- PASO 5: Construir y Guardar el JSON Final ---
sorted_labels = sorted(list(all_labels_set))
final_json_output_dict = {
    "classes": sorted_labels,
    "annotations": all_annotations_list
}

print(f"\n--- Resumen Final ---")
print(f"Total de textos procesados y añadidos al JSON: {len(all_annotations_list)}")
print(f"Clases/Etiquetas únicas encontradas: {sorted_labels}")

# Guardar en archivo JSON
try:
    print(f"\nGuardando JSON en: {output_json_file}")
    with open(output_json_file, 'w', encoding='utf-8') as f:
        # Usar ensure_ascii=False para correcta codificación de acentos/ñ
        # Usar indent=None para un archivo más compacto, o indent=4 para legibilidad
        json.dump(final_json_output_dict, f, ensure_ascii=False, indent=None)
    print(f"Archivo JSON guardado exitosamente.")

    # Opcional: Imprimir una pequeña muestra del JSON generado
    # print("\nVista previa del JSON generado (primeras 2 anotaciones):")
    # preview_dict = {"classes": sorted_labels, "annotations": all_annotations_list[:2]}
    # print(json.dumps(preview_dict, ensure_ascii=False, indent=4))

except Exception as json_err:
    print(f"ERROR al generar/guardar el archivo JSON: {json_err}")

print("\n--- Fin de la Generación de JSON ---")